# AAT6153 Chapter 1 — Hands-on 6
## Classifying Three Animals with Transfer Learning
**Slides:** Transfer Learning (slides 59–63)

In this notebook you will:
1. Re-use **InceptionV3**, pre-trained on ImageNet, as a frozen **feature extractor**.
2. Add a small classifier head and train only the head on the three-animals dataset.
3. Train the from-scratch CNN of Hands-on 5 side by side and see the **higher start, higher slope and higher asymptote** of transfer learning (slide 63).
4. *(Optional)* **Fine-tune** the top of the pre-trained network.

*Runtime: Runtime ▸ Change runtime type ▸ **T4 GPU** required.*

## Download the dataset and load the images (same as Hands-on 5)

In [ ]:
!gdown 1hO_sRQwh4MxqBXEuwGe-2KVIQr7OaE8g -O three_animals_dataset.zip
!unzip -q -o three_animals_dataset.zip && echo "A total of $(find three_animals_dataset/ -type f | wc -l) files have been extracted."

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.utils import image_dataset_from_directory, load_img, img_to_array
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

tf.keras.utils.set_random_seed(7)

train_dir = 'three_animals_dataset/train'
test_dir = 'three_animals_dataset/test'
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 5

train_ds, val_ds = image_dataset_from_directory(
    train_dir, validation_split=0.1, subset='both', seed=7,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical',
    interpolation='bicubic')
test_ds = image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical',
    interpolation='bicubic', shuffle=False)

class_names = train_ds.class_names
test_files = test_ds.file_paths
num_classes = len(class_names)
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(30 / 360, fill_mode='nearest'),
    layers.RandomTranslation(0.2, 0.2, fill_mode='nearest'),
    layers.RandomZoom(0.2, fill_mode='nearest'),
], name='data_augmentation')

---
## Load the pre-trained base model (slide 60)
* `include_top=False` drops InceptionV3's original 1000-class ImageNet classifier.
* We keep the network only up to the layer **`mixed7`**. Its features are general enough for our task and the model is smaller and faster.
* `trainable = False` **freezes** all the pre-trained weights.

In [ ]:
base_model = InceptionV3(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
feature_extractor = models.Model(base_model.input, base_model.get_layer('mixed7').output,
                                 name='inception_v3_to_mixed7')
feature_extractor.trainable = False

print("Layers in InceptionV3 up to mixed7:", len(feature_extractor.layers))
print("Output feature map shape:", feature_extractor.output.shape)

## Add a new classifier head
**Important:** every pre-trained model expects its inputs prepared exactly as during its ImageNet training. InceptionV3 expects pixels scaled to **[-1, 1]**, which is what `Rescaling(1/127.5, offset=-1)` does.
Calling the base with `training=False` keeps its Batch Normalization layers in inference mode, which matters if we fine-tune later.

In [ ]:
inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1)(x)        # InceptionV3 preprocessing: [0,255] → [-1,1]
x = feature_extractor(x, training=False)
x = layers.GlobalAveragePooling2D()(x)               # average each feature map to a single value
x = layers.Dense(512, activation='relu')(x)          # fully-connected layer with 512 hidden units
x = layers.Dropout(0.2)(x)                           # dropout rate of 0.2
outputs = layers.Dense(num_classes, activation='softmax')(x)   # softmax for multiclass output

tl_model = models.Model(inputs, outputs, name='inception_transfer')
tl_model.compile(optimizer=RMSprop(learning_rate=1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
tl_model.summary()

Note the difference between **total** and **trainable** parameters: millions of weights are re-used from ImageNet, and only the small head is trained.

## Train the classifier head

In [ ]:
tl_history = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

---
## Compare with training from scratch (slide 63)
We now train the LeNet-style CNN from Hands-on 5 on the **same data for the same number of epochs**.

In [ ]:
scratch_model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1 / 255.),
    layers.Conv2D(6, (5, 5), padding='same', activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(16, (5, 5), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dense(84, activation='relu'),
    layers.Dense(num_classes, activation='softmax'),
], name='from_scratch')
scratch_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
scratch_history = scratch_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, tl_history.history['val_accuracy'], 'g-o', label='Transfer learning (InceptionV3)')
plt.plot(epochs, scratch_history.history['val_accuracy'], 'r-o', label='CNN from scratch')
plt.xlabel('Epoch'); plt.ylabel('Validation accuracy'); plt.xticks(epochs)
plt.title('Higher start, higher slope, higher asymptote?')
plt.legend(); plt.grid(alpha=0.3); plt.show()

for name, m in [('Transfer learning', tl_model), ('From scratch', scratch_model)]:
    print(f"{name:18s} test accuracy: {m.evaluate(test_ds, verbose=0)[1]:.2%}")

---
## *(Optional)* Fine-tuning
Unfreeze the **top** blocks of the base model (from `mixed6` onward) and continue training with a **very small learning rate**, so the pre-trained features are adjusted gently instead of being destroyed.

In [ ]:
feature_extractor.trainable = True
fine_tune_from = [l.name for l in feature_extractor.layers].index('mixed6')
for layer in feature_extractor.layers[:fine_tune_from]:
    layer.trainable = False

tl_model.compile(optimizer=RMSprop(learning_rate=1e-5),       # must re-compile after changing trainable
                 loss='categorical_crossentropy', metrics=['accuracy'])
print("Trainable weight tensors:", len(tl_model.trainable_weights))

ft_history = tl_model.fit(train_ds, validation_data=val_ds,
                          epochs=EPOCHS + 3, initial_epoch=EPOCHS)

acc = tl_history.history['val_accuracy'] + ft_history.history['val_accuracy']
plt.plot(np.arange(1, len(acc) + 1), acc, 'g-o')
plt.axvline(EPOCHS + 0.5, color='grey', ls='--', label='start of fine-tuning')
plt.xlabel('Epoch'); plt.ylabel('Validation accuracy'); plt.legend(); plt.show()

---
## Evaluate the transfer learning model

In [ ]:
score = tl_model.evaluate(test_ds, verbose=0)
print(f'Loss: {score[0]:.4f}, accuracy: {score[1]:.2%}')

y_true = np.concatenate([np.argmax(y, axis=1) for _, y in test_ds])
y_pred = np.argmax(tl_model.predict(test_ds, verbose=0), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names, cmap='Greens')
plt.title('Confusion matrix (transfer learning)'); plt.show()

errors = [(test_files[i], class_names[y_true[i]], class_names[y_pred[i]])
          for i in np.where(y_true != y_pred)[0]]
print(f"{len(errors)} misclassified test images")
for path, true, pred in errors[:20]:
    print(f"  {path}  (true: {true}, predicted: {pred})")

## Save, reload and test on your own images

In [ ]:
tl_model.save('three_animals_transfer.keras')
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

model = tf.keras.models.load_model('three_animals_transfer.keras')
with open('class_names.json') as f:
    class_names = json.load(f)
print("Model loaded from disk. Classes:", class_names)

In [ ]:
from google.colab import files
uploaded = files.upload()

for fn in uploaded:
    img = load_img(fn, target_size=IMG_SIZE)
    x = np.expand_dims(img_to_array(img), axis=0)      # raw 0–255; preprocessing is inside the model
    probs = model.predict(x, verbose=0)[0]

    for name, p in zip(class_names, probs):
        print(f'{name:10}: {p: .3f}')
    predicted_class = class_names[probs.argmax()] if probs.max() > 0.5 else "None"

    plt.axis(False)
    plt.imshow(load_img(fn))
    plt.title(f'Predicted as {predicted_class}')
    plt.show()

---
## Try it yourself: other pre-trained backbones (slide 60)
Replace InceptionV3 with another model from `tf.keras.applications`. Use the **whole** base model (`feature_extractor = base_model`) and change the preprocessing layer to match:

| Backbone | Import | Preprocessing layer |
|---|---|---|
| VGG16 | `from tensorflow.keras.applications import VGG16` | `layers.Lambda(tf.keras.applications.vgg16.preprocess_input)` |
| ResNet50 | `from tensorflow.keras.applications import ResNet50` | `layers.Lambda(tf.keras.applications.resnet50.preprocess_input)` |
| MobileNetV2 | `from tensorflow.keras.applications import MobileNetV2` | `layers.Rescaling(1/127.5, offset=-1)` |
| EfficientNetB0 | `from tensorflow.keras.applications import EfficientNetB0` | *none*: rescaling is built into the model |

Record the number of parameters, training time per epoch and test accuracy for each. Which gives the best trade-off?

In [ ]:
# Use this cell for your experiments